# Blur / Quality Classifier Training

This notebook trains and evaluates two blur/quality classifiers:
- **BlurCNN** – a lightweight custom CNN for fast inference.
- **ResNet50** – a fine-tuned ResNet50 backbone for higher accuracy.

## Dataset setup

Download the **CERTH Image Blur Dataset** and place it under the following structure:

```
photo-declutterer/
└── data/
    └── certh/
        ├── train/
        │   ├── sharp/      ← undistorted images
        │   └── blurry/     ← artificially blurred / motion-blurred images
        └── val/
            ├── sharp/
            └── blurry/
```

Each leaf directory should contain the corresponding JPEG/PNG images.
Recommended split: **80 % train / 20 % val**.

> **Reference:** Mavridaki, E. & Mezaris, V. (2014). *No-Reference Blur Assessment in Natural Images Using Fourier Transform and Spatial Pyramids.* ICIP 2014.

In [ ]:
import sys
import logging
from pathlib import Path

# Make src/ importable when running from the notebooks/ directory
PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("blur_quality_training")
log.info("Environment ready. PROJECT_ROOT=%s", PROJECT_ROOT)

In [ ]:
from src.blur_quality import train_blur_cnn, train_resnet50

TRAIN_DIR = PROJECT_ROOT / "data" / "certh" / "train"
VAL_DIR   = PROJECT_ROOT / "data" / "certh" / "val"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Train BlurCNN ────────────────────────────────────────────────────────────
log.info("Training BlurCNN ...")
blur_cnn_model, blur_cnn_history = train_blur_cnn(
    train_dir=str(TRAIN_DIR),
    val_dir=str(VAL_DIR),
    epochs=20,
    batch_size=32,
    img_size=(224, 224),
    save_path=str(MODEL_DIR / "blur_cnn.pt"),
)
log.info("BlurCNN training complete.")

In [ ]:
# ── Train ResNet50 ───────────────────────────────────────────────────────────
log.info("Fine-tuning ResNet50 ...")
resnet_model, resnet_history = train_resnet50(
    train_dir=str(TRAIN_DIR),
    val_dir=str(VAL_DIR),
    epochs=15,
    batch_size=16,
    img_size=(224, 224),
    freeze_backbone_epochs=5,   # warm-up: only train the head for first 5 epochs
    save_path=str(MODEL_DIR / "resnet50_blur.pt"),
)
log.info("ResNet50 fine-tuning complete.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from src.blur_quality import evaluate_blur_model

CLASS_NAMES = ["blurry", "sharp"]

# ── Evaluate both models ─────────────────────────────────────────────────────
for label, model in [("BlurCNN", blur_cnn_model), ("ResNet50", resnet_model)]:
    y_true, y_pred = evaluate_blur_model(
        model=model,
        val_dir=str(VAL_DIR),
        img_size=(224, 224),
        batch_size=32,
    )

    print(f"\n{'='*60}")
    print(f"  {label} — Classification Report")
    print(f"{'='*60}")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred,
        display_labels=CLASS_NAMES,
        cmap="Blues",
        ax=ax,
    )
    ax.set_title(f"{label} — Confusion Matrix")
    plt.tight_layout()
    plt.savefig(str(MODEL_DIR / f"{label.lower()}_confusion_matrix.png"), dpi=150)
    plt.show()

log.info("Evaluation complete. Figures saved to %s", MODEL_DIR)